## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import plotly.io as pio
pio.renderers.default = 'jupyterlab' # or 'notebook_connected'
from importlib import reload

from math import *
import numpy as np
import radiometry_test

reload(radiometry_test)

<module 'radiometry_test' from '/Users/espillar/Desktop/vibevolts/radiometry_test.py'>

In [3]:
def amag(m):  return(10**(-0.4 * m))
def mag(f): return( -2.5 * log10(f))
print(amag(5))
print(mag(100))

0.01
-5.0


V band solar brightness is -26.74 accounding to gemini, radiometry_data has it as -26.78. Agreement. 

Gemini makes this as $5.1 \times 10^{20}$ photons per second per square meter.  Using the constants in my head I get pretty good agreement- although a little off!

In [4]:
print(amag(-26.74) * 866000 * 10000)

4.300489503759908e+20


## Code: 
lambertian_satellite_magnitude(rad, dist, alb, phs_deg), 

vbandphots(mag)

In [5]:


def lambertian_satellite_magnitude(radius, distance, albedo, phase_angle_deg):
    """
    Calculates the apparent V-band magnitude of a spherical satellite.
    """
    m_sun = -26.74
    alpha_rad = np.radians(phase_angle_deg)
    
    # 1. Get the Lambertian Phase Function value
    # Note: We divide by the peak value at alpha=0 (2/(3*pi)*pi) 
    # to use it as a scaling factor between 0 and 1.
    phi = (1 / np.pi) * ((np.pi - alpha_rad) * np.cos(alpha_rad) + np.sin(alpha_rad))
    
    # 2. Calculate the brightness ratio
    # (Albedo * Area_ratio * Phase)
    # The (2/3) comes from the integration of a diffuse sphere's brightness
    brightness_ratio = albedo * (2/3) * (radius**2 / distance**2) * phi
    
    # 3. Convert to magnitude
    mag = m_sun - 2.5 * np.log10(brightness_ratio)
    
    return mag

# Example: A 2-meter radius satellite at 550km altitude (LEO) 
# at a 30-degree phase angle with an albedo of 0.5
sat_mag = lambertian_satellite_magnitude(radius=2, distance=100000000, albedo=0.2, phase_angle_deg=0)

print(f"Satellite Apparent Magnitude: {sat_mag:.2f}")

Satellite Apparent Magnitude: 13.94


Converted this to LaTeX and compared algorithm with Cognion, it looks good.

### Checking that our memory and gemini agree on the magnitudes to photons/cm^2
vbandphots is the function

In [6]:
def vbandphots(magnitude):
    ''' The output here will be in photons per cm2 per second'''
    p = 866000 * amag(magnitude)
    return(p) 

In [7]:
def v_mag_to_photon_flux(v_mag, band_width_angstroms=890):
    """
    Converts a V-band magnitude to photon flux (photons / cm^2 / s).
    
    Physics Constants based on Bessell (1979) / Vega system:
    - V-band Effective Wavelength: ~5500 Angstroms
    - V-band Zero Point (Flux Density at V=0): ~3.63e-9 erg / cm^2 / s / A
    - V-band FWHM (Bandwidth): ~890 Angstroms (default)
    
    Args:
        v_mag (float): The apparent V magnitude.
        band_width_angstroms (float): The filter width to integrate over. 
                                      Defaults to 890 A (standard Johnson V).
                                      
    Returns:
        float: Integrated photon flux (photons / cm^2 / s)
    """
    
    # 1. Constants
    # Zero point flux density in energy units (erg / cm^2 / s / A)
    # Source: Bessell (1979) for Vega
    F_0_energy = 3.63e-9 
    
    # Energy of a single photon at 5500 Angstroms
    # E = hc / lambda
    h = 6.626e-27 # Planck constant (erg * s)
    c = 2.998e10  # Speed of light (cm / s)
    wavelength_cm = 5500 * 1e-8
    
    E_photon = (h * c) / wavelength_cm # approx 3.61e-12 erg
    
    # 2. Convert Zero Point Energy Flux to Photon Flux Density
    # (photons / cm^2 / s / A)
    N_0_density = F_0_energy / E_photon 
    
    # 3. Calculate Flux Density for the specific magnitude
    # Formula: F = F0 * 10^(-0.4 * m)
    flux_density = N_0_density * 10**(-0.4 * v_mag)
    
    # 4. Integrate over the bandwidth to get total photons per second per cm^2
    total_photon_flux = flux_density * band_width_angstroms
    
    return float(total_photon_flux)

In [8]:
print(vbandphots(20), "  ", v_mag_to_photon_flux(20))

0.008660000000000001    0.008944915888185443


Which I think is good  agreement.  

### Function to get the photoelectrons 
photoelectrons and backgroundshotnoise

In [9]:
def photoelectrons(r, range, albedo, phaseangle, itime, aper, qe):
    """ aper radios in cm """
    electrons = vbandphots(lambertian_satellite_magnitude(r, range, albedo, phaseangle) )  * ( # phots per cm2 per sec
        itime * pi * aper**2 * qe ) # times itime pi r^2 times qe
    return electrons

In [10]:
def backgroundshotnoise(pixsizeasec, aperrad, backmagV, itime,qe):
    """we are assuming V band for the background
    backmagV is magnitudes per square arcsecond
    pixsizeasec is one pixel side units arcseconds
    aperrad is in cm
    This is ONLY the blp"""
    pe = vbandphots(backmagV) * itime * pixsizeasec**2 * pi * aperrad**2 * qe
    return(sqrt(pe))

### Electrons received, read noise, SNR using tools written here. 

In [11]:
r = 0.5 # 1m radius satellite, 
satrange = 1e6 # 100000000 m
albedo = 0.2 # 0.2 albedo
phaseangle = 0 #  0 degree phase angle
itime = 1 #Integration Time
aperrad = 50 # 50 cm aperture
qe = 1 # Net conversions of incident photons to photoelectrons
pixsizeasec = 1 # Pixel size in arcsec
backmagV  = 23 # Magnitudes per square arcsecond


print(lambertian_satellite_magnitude(r, satrange, albedo, phaseangle), "  satellite magnitude")
signal = photoelectrons(r, satrange, albedo, phaseangle, itime, aperrad, qe)
noise = backgroundshotnoise(pixsizeasec, aperrad, backmagV, itime, qe)
print(" Signal,           Noise,               SNR")
print(f"{signal:.2e}", noise, signal/noise)

6.952803136799158   satellite magnitude
 Signal,           Noise,               SNR
1.13e+07 2.0715903852811555 5434788.30236127


## VV Code Tests

This signal of 9006 agrees with the MMa estimate 

In [5]:
fig = radiometry_test.demoFixed()

targets are 1e8 m diameter 1 along line of sight
targets are 1e8 m diameter 2
targets are 1e9 m diameter 1
targets are 1e8 m diameter 1 perp to line of site
--- Running scandetector ---
sun, space, sky 
    4.529e+20, 2.360e+11, 6.499e+11
unit_vectors1, 2  [[-1.  0.  0.]
 [-1.  0.  0.]
 [-1.  0.  0.]
 [-1.  0.  0.]]     [[-1.e+00 -0.e+00 -0.e+00]
 [-1.e+00 -0.e+00 -0.e+00]
 [-1.e+00 -0.e+00 -0.e+00]
 [ 1.e-06 -1.e+00 -0.e+00]]
dot product  [ 1.e+00  1.e+00  1.e+00 -1.e-06]
angles  [0.         0.         0.         1.57079733]

--- Debug Info: lambertiansphere ---
Effective Cross Section:  [0.10471976 0.41887902 0.10471976 0.03333328]
Index Phase Angle (rad)  Albedo     Radius (m)   Base Brightness    Emitted Brightness    
------------------------------------------------------------------------------------------
0     0.0000e+00         0.2000     5.0000e-01   4.5289e+20         4.7426e+19            
1     0.0000e+00         0.2000     1.0000e+00   4.5289e+20         1.8970e+20       

In [8]:
print( 1 - (4.718 / 14.82))

0.6816464237516869


# Pencil Tests

## Satellite Brightness

### 5m Geo Brightness
This is a little bright by memory, but I think OK.  Things are close

In [14]:
satrad = 250 # cm
telerad = 0.005 # m - a 1 cm telescope
sunmag = -26.74 # V mag
range = 40e6 # Geo

phots = (amag(sunmag) *  # solar magnitude 
866000 * # now in photons/cm2/sec
pi * satrad**2 *  # photons/sec
0.2 *              # photons/sec
pi * telerad**2  /  #photons m2/sec
 (4 * pi * range**2) )   #photons/sec
print(mag( phots/866000 ))

12.795450199024149


### m telescope m satellite

In [15]:
satrad = 50 # cm
telerad = 0.5 # m
sunmag = -26.74 # V mag
range = 1e6

phots = (amag(sunmag) *  # solar magnitude 
866000 * # now in photons/cm2/sec
pi * satrad**2 *  # photons/sec
0.2 *              # photons/sec
pi * telerad**2  /  #photons m2/sec
 (4 * pi * range**2) )   #photons/sec
print(f"{phots:.2e}")


4.22e+06


### m telescope m satellite 1e8 away

In [16]:
satrad = 50 # cm
telerad = 0.5 # m
sunmag = -26.74 # V mag
range = 1e8

phots = (amag(sunmag) *  # solar magnitude 
866000 * # now in photons/cm2/sec
pi * satrad**2 *  # photons/sec
0.2 *              # photons/sec
pi * telerad**2  /  #photons m2/sec
 (4 * pi * range**2) )   #photons/sec
print(f"{phots:.2e}")


4.22e+02


In [17]:
print(amag(sunmag) * 866000) # solar magnitude 

4.300489503759908e+16


## Sky Brightness

These number check, although why is the pixel size less than 1 arcsec squared? Fixed

In [18]:
omegaarcsec = 1.84e-11 * (180 * 3600/ pi)**2 
print(omegaarcsec)

0.7828311334492005


In [19]:
sky = amag(23) * 866000 # photons/cm2/sec/as2
sky = sky * pi * 50 *50         #  area of the telescope in cm2
sky = sky * 1                       # 1 sec of integration
sky = sky * 1          # pixel area in as2
print(sqrt(sky))

2.0715903852811555
